# Step 5 — CatBoost control with exact-feature-group-disjoint validation

**Purpose:** Establish a strong, **single-row tabular** intrusion-detection control before reproducing SCV-IDS or proposing a new image-based network. This is training-only validation, not the official benchmark test score. Evaluation uses **ten-class macro-F1**, with class-level recall and benign false-positive rate.

**Integrity:** All training comes from the **verified official train partition** (although the mirror calls it `unsw_test.csv`). A group-stratified fold prevents identical 42-feature vectors from crossing the internal train/validation boundary, even when their labels conflict.

**Kaggle settings:** GPU accelerator enabled and Internet enabled. You already have the fine-grained GitHub secret **`pushRare`**; ensure access is enabled. No secret value is printed or committed.

**Artifact storage:** small configuration, aggregate metrics, confusion matrix, class table and checkpoint hash sync to GitHub. The `.cbm` checkpoint stays in the Kaggle output directory: use **Save Version → Save & Run All** to preserve it. No official testing CSV is read for model fitting/prediction.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path("/kaggle/working/rare-attack-ids-step05")
URL = "https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", URL, str(REPO)], check=True)

git_sha = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                                  text=True).strip()
print("Research commit:", git_sha)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                "--format=csv,noheader"], check=True)

In [ ]:
try:
    import catboost
    print("CatBoost:", catboost.__version__)
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "catboost"],
                   check=True)
    import catboost
    print("CatBoost installed:", catboost.__version__)

import sklearn
print("scikit-learn:", sklearn.__version__)

In [ ]:
CONFIG = REPO / "configs" / "step05_catboost_groupval.json"
OUTPUT = REPO / "results" / "CATBOOST-GROUPVAL-001"
CHECKPOINTS = Path("/kaggle/working/checkpoints")

environment = os.environ.copy()
environment["PYTHONPATH"] = str(REPO / "src")
subprocess.run([
    sys.executable, str(REPO / "scripts" / "run_catboost_groupval.py"),
    "--repo", str(REPO), "--config", str(CONFIG),
    "--output", str(OUTPUT), "--checkpoints", str(CHECKPOINTS)],
    cwd=str(REPO), env=environment, check=True)
print((OUTPUT / "run_report.md").read_text(encoding="utf-8"))

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("pushRare")
except Exception:
    token = None

if not token:
    print("VALIDATION RESULTS NOT SYNCED: allow access to Kaggle pushRare secret.")
else:
    safe_env = os.environ.copy()
    safe_env["GITHUB_TOKEN"] = token
    try:
        subprocess.run([
            sys.executable, str(REPO / "scripts" / "sync_groupval_results.py"),
            "--directory", str(OUTPUT)],
            cwd=str(REPO), env=safe_env, check=True)
    finally:
        safe_env.pop("GITHUB_TOKEN", None)
        del token

## What to return

After the final cell, send the **VERIFIED BASELINE SYNC COMMIT** link. If the run fails, share the **first Python traceback** and we will correct the committed implementation. Do not alter the official test partition or skip the exact-feature-group validation gate.

We will then independently inspect the validation metrics and determine the next architectural comparison; no final-paper performance conclusion can be drawn from this step alone.